# Telecom Customer Churn Prediction

This notebook demonstrates the end-to-end pipeline for predicting customer churn in a telecom network. We explore the dataset, perform preprocessing and feature scaling, train a Random Forest Classifier, and serialize the trained model and scaler for deployment.

## 1. Import Libraries & Load Data

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

sns.set_theme(style="whitegrid")

# Load dataset
dataset_url = "https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(dataset_url)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Churn class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Distribution of Customer Churn')
plt.show()

In [ ]:
# Churn relationships with key categorical variables
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='Contract', hue='Churn', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Churn by Contract Type')
sns.countplot(x='InternetService', hue='Churn', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Churn by Internet Service Type')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
# Drop redundant columns
df_clean = df.drop(columns=['customerID'])

# Clean TotalCharges (convert to numeric, fill blank charges with 0.0)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0.0)

# Map binary features to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies', 
               'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_clean[col] = df_clean[col].apply(lambda x: 1 if x == 'Yes' else 0)

df_clean['gender'] = df_clean['gender'].apply(lambda x: 1 if x == 'Female' else 0)

# One-Hot Encode multi-class categories
multi_cat_cols = ['InternetService', 'Contract', 'PaymentMethod']
df_clean = pd.get_dummies(df_clean, columns=multi_cat_cols, drop_first=True)

# Convert bool columns to int
bool_cols = df_clean.select_dtypes(include=['bool']).columns
df_clean[bool_cols] = df_clean[bool_cols].astype(int)

df_clean.head()

## 4. Train-Test Split & Scaling

In [ ]:
# Split X and y
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical attributes
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size: {X_test.shape[0]} samples")

## 5. Model Training & Serialization

In [ ]:
# Initialize and train the Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_clf.fit(X_train_scaled, y_train)

print(f"Random Forest Test Accuracy: {rf_clf.score(X_test_scaled, y_test):.4f}")

In [ ]:
# Create models directory and save artifacts
os.makedirs('models', exist_ok=True)

joblib.dump(rf_clf, 'models/model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

columns = list(X_train.columns)
with open('models/columns.json', 'w') as f:
    json.dump(columns, f)

print("Model artifacts saved successfully in 'models/'!")